# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ziadhamouda370-beep/flyrank-ml/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*I will use a transparent refresh-priority rule that combines four observable signals: search visibility, freshness risk, position opportunity, and content depth gap. The score ranks content for human review rather than making an automatic publishing decision. Reason codes will explain common review opportunities such as stale visible pages, declining pages with demand, thin visible pages, page-one decay risk, low CTR on visible pages, and low engagement on visible pages.

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np
from pathlib import Path

url = "https://raw.githubusercontent.com/ziadhamouda370-beep/flyrank-ml/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(url)

def percentile_rank(s):
    return s.rank(method="average", pct=True).fillna(0)

def normalize(s):
    s = s.astype(float)
    minimum = s.min()
    maximum = s.max()
    if maximum == minimum:
        return pd.Series(0.0, index=s.index)
    return (s - minimum) / (maximum - minimum)

print("Rows:", len(df))
print("Columns:", len(df.columns))
print("Baseline rule: visibility + freshness + position opportunity + depth gap")

Rows: 30000
Columns: 44
Baseline rule: visibility + freshness + position opportunity + depth gap


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*I will calculate a 0-to-1 baseline refresh score for every content item, rank all items from highest to lowest priority, assign a suggested action, and attach reason codes. The output will be saved as work/outputs/baseline_action_score.csv for later comparison with the ML model.

In [15]:
# 1. Visibility: higher impressions = higher visibility
df["visibility_score"] = percentile_rank(
    np.log1p(df["impressions_90d"])
)

# 2. Freshness risk: older updates = higher refresh risk
df["freshness_risk_score"] = percentile_rank(
    df["days_since_last_update"]
)

# 3. Position opportunity:
# Pages with better positions and meaningful visibility get more opportunity score
df["position_opportunity_score"] = (
    (1 - normalize(
        df["avg_position"].clip(lower=1, upper=50)
    ))
    * df["visibility_score"]
    * (df["avg_position"] > 0).astype(int)
)

# 4. Depth gap:
# Shorter content with meaningful visibility gets more review priority
df["depth_gap_score"] = (
    (1 - percentile_rank(df["word_count"]))
    * df["visibility_score"]
)

# Final transparent baseline score
df["baseline_refresh_score"] = (
    0.40 * df["visibility_score"]
    + 0.30 * df["freshness_risk_score"]
    + 0.25 * df["position_opportunity_score"]
    + 0.05 * df["depth_gap_score"]
).clip(0, 1)


# Reason codes
def reason_codes(row):
    reasons = []

    if row["days_since_last_update"] >= 180 and row["impressions_90d"] >= 500:
        reasons.append("stale_visible_page")

    if (
        str(row["trend_direction"]).lower() == "down"
        and row["impressions_90d"] >= 100
    ):
        reasons.append("declining_with_demand")

    if (
        row["word_count"] > 0
        and row["word_count"] < 1200
        and row["impressions_90d"] >= 250
    ):
        reasons.append("thin_visible_page")

    if (
        row["avg_position"] > 0
        and row["avg_position"] <= 10
        and row["content_age_days"] >= 180
    ):
        reasons.append("page_one_decay_risk")

    if (
        row["impressions_90d"] >= 500
        and 0 < row["avg_position"] <= 20
        and row["ctr"] < 0.5
    ):
        reasons.append("low_ctr_visible_page")

    if (
        row["sessions_90d"] >= 30
        and (
            (row["engagement_rate"] > 0 and row["engagement_rate"] < 30)
            or
            (row["scroll_rate"] > 0 and row["scroll_rate"] < 30)
        )
    ):
        reasons.append("low_engagement_visible_page")

    if not reasons:
        reasons.append("general_refresh_review")

    return "|".join(reasons)


df["reason_codes"] = df.apply(reason_codes, axis=1)


# Suggested action
def suggested_action(row):
    reasons = set(str(row["reason_codes"]).split("|"))

    if "thin_visible_page" in reasons:
        return "expand_and_refresh"

    if "low_ctr_visible_page" in reasons:
        return "refresh_and_review_ctr"

    if (
        "stale_visible_page" in reasons
        or "declining_with_demand" in reasons
    ):
        return "refresh"

    return "monitor"


df["suggested_action"] = df.apply(
    suggested_action,
    axis=1
)


# Rank everything
df["baseline_rank"] = (
    df["baseline_refresh_score"]
    .rank(method="first", ascending=False)
    .astype(int)
)

# Sort
df = df.sort_values("baseline_rank")


# Required output
output_columns = [
    "content_id",
    "baseline_rank",
    "baseline_refresh_score",
    "visibility_score",
    "freshness_risk_score",
    "position_opportunity_score",
    "depth_gap_score",
    "reason_codes",
    "suggested_action",
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "avg_position",
    "ctr",
    "engagement_rate",
    "scroll_rate",
    "content_age_days",
    "days_since_last_update",
    "word_count"
]

output = df[output_columns]

output_path = Path("work/outputs/baseline_action_score.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

output.to_csv(output_path, index=False)

print("Saved:", output_path)
print("Rows scored:", len(output))
print("Top score:", round(output["baseline_refresh_score"].max(), 3))
print("Median score:", round(output["baseline_refresh_score"].median(), 3))

print("\nTop 20:")
display(output.head(20))

Saved: work/outputs/baseline_action_score.csv
Rows scored: 30000
Top score: 0.948
Median score: 0.446

Top 20:


,content_id,baseline_rank,baseline_refresh_score,visibility_score,freshness_risk_score,position_opportunity_score,depth_gap_score,reason_codes,suggested_action,impressions_90d,clicks_90d,sessions_90d,avg_position,ctr,engagement_rate,scroll_rate,content_age_days,days_since_last_update,word_count
21565,content_9532f197bbc8,1,0.947603,0.999633,0.8432,0.979233,0.999633,declining_with_demand|page_one_decay_risk|low_...,refresh,309192,2689,1098,2.0,0.87,8.01,28.75,445,104,NaN
4644,content_4d1fe5b32dc2,2,0.941268,0.994167,0.8432,0.963733,0.994167,page_one_decay_risk|low_engagement_visible_page,monitor,97999,512,549,2.5,0.52,7.47,13.15,329,104,NaN
18954,content_07f2e7a6f38a,3,0.940461,0.994467,0.8432,0.959965,0.994467,page_one_decay_risk|low_engagement_visible_page,monitor,101078,856,780,2.7,0.85,2.05,4.60,313,104,NaN
17400,content_e5ae436f9a16,4,0.939997,0.996000,0.8432,0.955347,0.996000,page_one_decay_risk|low_ctr_visible_page|low_e...,refresh_and_review_ctr,117741,533,522,3.0,0.45,7.09,12.60,421,104,NaN
9348,content_3430a8b94511,5,0.939963,0.998167,0.8432,0.951314,0.998167,page_one_decay_risk|low_ctr_visible_page|low_e...,refresh_and_review_ctr,152617,440,534,3.3,0.29,6.18,11.04,329,104,NaN
25409,content_cbd93118300b,6,0.939665,0.997733,0.8432,0.950901,0.997733,declining_with_demand|page_one_decay_risk|low_...,refresh_and_review_ctr,145292,662,535,3.3,0.46,1.87,5.38,313,104,NaN
18458,content_9c195417f6ef,7,0.939353,0.991400,0.8432,0.961051,0.991400,page_one_decay_risk|low_engagement_visible_page,monitor,79146,574,515,2.5,0.73,1.55,2.79,313,104,NaN
13306,content_ba2acb4ebd04,8,0.938024,0.997567,0.8432,0.944635,0.997567,page_one_decay_risk|low_engagement_visible_page,monitor,142072,1185,1147,3.6,0.83,1.92,5.08,362,104,NaN
28354,content_79b25654070a,9,0.937766,0.997933,0.8432,0.942945,0.997933,page_one_decay_risk|low_ctr_visible_page|low_e...,refresh_and_review_ctr,148737,711,619,3.7,0.48,2.26,3.46,257,104,NaN
8275,content_adddad39251c,10,0.937520,0.996833,0.8432,0.943940,0.996833,page_one_decay_risk|low_engagement_visible_page,monitor,129239,711,688,3.6,0.55,3.92,6.32,329,104,NaN


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*I will review the top 20 ranked items as a human-review queue. Each item will have a suggested action, reason codes, and a confidence note. The ranking is directional, so a high score does not mean the item definitely needs a refresh. A recommendation could be wrong because of seasonality, low traffic volume, measurement changes, or context that is not represented in the available fields.

In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top20 = output.head(20).copy()

top20["confidence_note"] = np.select(
    [
        top20["baseline_refresh_score"] >= 0.70,
        top20["baseline_refresh_score"] >= 0.40
    ],
    [
        "Higher priority; verify before action.",
        "Moderate priority; verify before action."
    ],
    default="Lower priority; monitor and verify."
)

top20["what_could_make_it_wrong"] = (
    "Seasonality, low volume, measurement changes, or missing context"
)

review_columns = [
    "baseline_rank",
    "content_id",
    "baseline_refresh_score",
    "suggested_action",
    "reason_codes",
    "confidence_note",
    "what_could_make_it_wrong"
]

display(top20[review_columns])

,baseline_rank,content_id,baseline_refresh_score,suggested_action,reason_codes,confidence_note,what_could_make_it_wrong
21565,1,content_9532f197bbc8,0.947603,refresh,declining_with_demand|page_one_decay_risk|low_...,Higher priority; verify before action.,"Seasonality, low volume, measurement changes, ..."
4644,2,content_4d1fe5b32dc2,0.941268,monitor,page_one_decay_risk|low_engagement_visible_page,Higher priority; verify before action.,"Seasonality, low volume, measurement changes, ..."
18954,3,content_07f2e7a6f38a,0.940461,monitor,page_one_decay_risk|low_engagement_visible_page,Higher priority; verify before action.,"Seasonality, low volume, measurement changes, ..."
17400,4,content_e5ae436f9a16,0.939997,refresh_and_review_ctr,page_one_decay_risk|low_ctr_visible_page|low_e...,Higher priority; verify before action.,"Seasonality, low volume, measurement changes, ..."
9348,5,content_3430a8b94511,0.939963,refresh_and_review_ctr,page_one_decay_risk|low_ctr_visible_page|low_e...,Higher priority; verify before action.,"Seasonality, low volume, measurement changes, ..."
25409,6,content_cbd93118300b,0.939665,refresh_and_review_ctr,declining_with_demand|page_one_decay_risk|low_...,Higher priority; verify before action.,"Seasonality, low volume, measurement changes, ..."
18458,7,content_9c195417f6ef,0.939353,monitor,page_one_decay_risk|low_engagement_visible_page,Higher priority; verify before action.,"Seasonality, low volume, measurement changes, ..."
13306,8,content_ba2acb4ebd04,0.938024,monitor,page_one_decay_risk|low_engagement_visible_page,Higher priority; verify before action.,"Seasonality, low volume, measurement changes, ..."
28354,9,content_79b25654070a,0.937766,refresh_and_review_ctr,page_one_decay_risk|low_ctr_visible_page|low_e...,Higher priority; verify before action.,"Seasonality, low volume, measurement changes, ..."
8275,10,content_adddad39251c,0.937520,monitor,page_one_decay_risk|low_engagement_visible_page,Higher priority; verify before action.,"Seasonality, low volume, measurement changes, ..."


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*Weak picks are high-ranked items whose score may be driven by unstable or incomplete signals. In particular, percentage-free percentile scores can still prioritize pages with limited context, while seasonality or measurement changes may make a page look like a refresh opportunity. I will treat these as review candidates rather than confirmed actions. The baseline does not use trend_direction or trend_pct as scoring features, and it does not use future windows.

In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Find potentially weak high-ranked picks
weak_picks = output[
    (output["baseline_rank"] <= 20) &
    (
        (output["impressions_90d"] < 100) |
        (output["sessions_90d"] < 10) |
        (output["word_count"] <= 0)
    )
]

print("Potential weak picks in Top-20:", len(weak_picks))

if len(weak_picks) > 0:
    display(
        weak_picks[
            [
                "baseline_rank",
                "content_id",
                "baseline_refresh_score",
                "suggested_action",
                "reason_codes",
                "impressions_90d",
                "sessions_90d",
                "word_count"
            ]
        ]
    )


# Leakage check
score_features = [
    "visibility_score",
    "freshness_risk_score",
    "position_opportunity_score",
    "depth_gap_score"
]

forbidden_fields = [
    "trend_direction",
    "trend_pct"
]

print("\nLeakage check")
print("Score features:", score_features)
print(
    "Forbidden label fields used:",
    [c for c in forbidden_fields if c in score_features]
)
print("Future-window fields used: none")

Potential weak picks in Top-20: 0

Leakage check
Score features: ['visibility_score', 'freshness_risk_score', 'position_opportunity_score', 'depth_gap_score']
Forbidden label fields used: []
Future-window fields used: none


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.